# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
import os
import json
import chromadb
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
from tavily import TavilyClient
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
from openai import OpenAI

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
openai_client = OpenAI(api_key=OPENAI_API_KEY)


In [ ]:
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "games_collection"

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
def retrieve_game(query: str, n_results: int = 3) -> list[dict]:
    """
    Semantic search: Finds the most relevant results in the vector DB.

    Args:
        query: A question about the game industry.
        n_results: Number of results to retrieve.

    Returns:
        A list of dictionaries. Each dictionary contains:
        - Platform: platform of the game
        - Name: name of the game
        - YearOfRelease: release year
        - Description: additional details
        - document: indexed text used in the vector database
        - distance: semantic distance from the query
    """

    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    retrieved_games = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for document, metadata, distance in zip(documents, metadatas, distances):
        retrieved_games.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "document": document,
            "distance": distance
        })

    return retrieved_games

In [ ]:
results = retrieve_game("Which game was released for Nintendo 64?")

for game in results:
    print(f"Name: {game['Name']}")
    print(f"Platform: {game['Platform']}")
    print(f"Year: {game['YearOfRelease']}")
    print(f"Description: {game['Description']}")
    print(f"Distance: {game['distance']}")
    print("-" * 50)

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful enough to answer the user's question.")

    description: str = Field(description="Detailed explanation of why the documents are or are not useful.")

In [ ]:
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> EvaluationReport:
    """
    LLM-as-a-judge evaluation tool.

    Instead of using only Chroma distance + a fixed threshold, this function asks
    an LLM to judge whether the retrieved documents are useful enough to answer
    the user's question.
    """

    if not retrieved_docs:
        return EvaluationReport(
            useful=False,
            description=(
                "No documents were retrieved from the vector database. "
                "The agent should use web search or another external source."
            )
        )

    docs_for_prompt = []
    for index, doc in enumerate(retrieved_docs, start=1):
        docs_for_prompt.append({
            "rank": index,
            "name": doc.get("Name"),
            "platform": doc.get("Platform"),
            "year": doc.get("YearOfRelease"),
            "description": doc.get("Description"),
            "document": doc.get("document"),
            "distance": doc.get("distance")
        })

    prompt = f"""
You are an evaluation tool for a RAG video game research agent.

Your job:
Decide whether the retrieved documents are useful enough to answer the user's question.

User question:
{question}

Retrieved documents:
{json.dumps(docs_for_prompt, indent=2)}

Return ONLY valid JSON with this exact structure:
{{
  "useful": true or false,
  "description": "short explanation of why the retrieved documents are or are not useful"
}}

Rules:
- useful should be true only if the retrieved documents directly help answer the question.
- useful should be false if the documents are unrelated, too vague, or do not contain enough information.
- Do not judge only by the Chroma distance. Use the meaning of the question and documents.
"""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a strict RAG retrieval evaluator. Always return valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    evaluation_json = json.loads(response.choices[0].message.content)

    return EvaluationReport(
        useful=bool(evaluation_json["useful"]),
        description=evaluation_json["description"]
    )


In [ ]:
question = "Which game was released for Nintendo 64?"

retrieved_docs = retrieve_game(question)

evaluation = evaluate_retrieval(question, retrieved_docs)

print("Useful:", evaluation.useful)
print("Description:", evaluation.description)

#### Game Web Search Tool

In [ ]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


def game_web_search(question: str, max_results: int = 3) -> list[dict]:

    search_query = f"video game industry {question}"

    response = tavily_client.search(query=search_query, search_depth="basic",  max_results=max_results)

    web_results = []

    for result in response.get("results", []):
        web_results.append({
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        })

    return web_results

In [ ]:
question = "Was Mortal Kombat X released for PlayStation 5?"

web_results = game_web_search(question)

for result in web_results:
    print("Title:", result["title"])
    print("URL:", result["url"])
    print("Content:", result["content"])
    print("-" * 50)

### Agent

In [ ]:
class AgentState(str, Enum):
    START = "start"
    RETRIEVE = "retrieve"
    EVALUATE = "evaluate"
    WEB_SEARCH = "web_search"
    RESPOND = "respond"

class AgentResponse(BaseModel):

    question: str = Field(description="Original user question.")
    answer: str = Field(description="Final answer to the user.")
    source: str = Field(description="Source used to answer: vector_database or web_search.")
    tool_trace: list[str] = Field(description="List of tools used by the agent.")
    citation: Optional[str] = Field(default=None, description="Citation or source URL, if available.")
    retrieved_docs: Optional[list[dict]] = Field(default=None, description="Documents retrieved from the vector database.")
    web_results: Optional[list[dict]] = Field(default=None, description="Results retrieved from web search.")


In [ ]:
class UdaPlayAgent:

    def __init__(self):
        self.state = AgentState.START
        self.conversation_history = []

    def answer_from_retrieval(self, question: str, retrieved_docs: list[dict]) -> str:

        best_doc = retrieved_docs[0]

        name = best_doc.get("Name")
        platform = best_doc.get("Platform")
        year = best_doc.get("YearOfRelease")
        description = best_doc.get("Description")

        return (
            f"Based on the local game database, the most relevant result is "
            f"{name}, released for {platform} in {year}. "
            f"{description}"
        )

    def answer_from_web(self, question: str, web_results: list[dict]) -> tuple[str, Optional[str]]:

        if not web_results:
            return (
                "I could not find enough information in the local database or through web search "
                "to answer this question confidently.",
                None
            )

        best_result = web_results[0]

        title = best_result.get("title")
        content = best_result.get("content")
        url = best_result.get("url")

        answer = (
            f"Based on web search, the most relevant source I found is '{title}'. "
            f"{content}"
        )

        return answer, url

    def run(self, question: str) -> AgentResponse:

        tool_trace = []

        self.state = AgentState.START
        tool_trace.append("Agent started.")

        self.state = AgentState.RETRIEVE
        tool_trace.append("Using retrieve_game to search the local vector database.")
        retrieved_docs = retrieve_game(question)

        self.state = AgentState.EVALUATE
        tool_trace.append("Using evaluate_retrieval to check if retrieved documents are useful.")
        evaluation = evaluate_retrieval(question, retrieved_docs)

        if evaluation.useful:
            self.state = AgentState.RESPOND
            tool_trace.append("Retrieved documents were considered useful.")
            tool_trace.append("Answering using the local vector database.")

            answer = self.answer_from_retrieval(question, retrieved_docs)

            response = AgentResponse(
                question=question,
                answer=answer,
                source="vector_database",
                tool_trace=tool_trace,
                citation=retrieved_docs[0].get("document"),
                retrieved_docs=retrieved_docs,
                web_results=None
            )

        else:
            self.state = AgentState.WEB_SEARCH
            tool_trace.append("Retrieved documents were not useful enough.")
            tool_trace.append("Using game_web_search to search external sources.")

            web_results = game_web_search(question)

            self.state = AgentState.RESPOND
            tool_trace.append("Answering using web search results.")

            answer, citation = self.answer_from_web(question, web_results)

            response = AgentResponse(
                question=question,
                answer=answer,
                source="web_search",
                tool_trace=tool_trace,
                citation=citation,
                retrieved_docs=retrieved_docs,
                web_results=web_results
            )

        self.conversation_history.append({
            "question": question,
            "answer": response.answer,
            "source": response.source
        })

        return response

    def show_history(self):
        return self.conversation_history

In [ ]:
agent = UdaPlayAgent()

In [ ]:
response = agent.run("When Pokémon Gold and Silver was released?")

print("Question:", response.question)
print("Answer:", response.answer)
print("Source:", response.source)
print("Citation:", response.citation)
print("\nTool Trace:")
for step in response.tool_trace:
    print("-", step)

In [ ]:
agent = UdaPlayAgent(session_id="demo_two_turns")

first_response = agent.run("Which Mario game was the first 3D platformer?")
second_response = agent.run("When was it released?")

for response in [first_response, second_response]:
    print("=" * 80)
    print("Question:", response.question)
    print("Answer:", response.answer)
    print("Source:", response.source)
    print("Citation:", response.citation)

    print("\nTool Trace:")
    for step in response.tool_trace:
        print("-", step)

    print()


In [ ]:
agent.show_history()

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes